# 모듈 (3) Self-Refine — 자기 수정(Self-Correction) 에이전트
## 에이전트 거버넌스 (Governance)

---

### 학습 목표
1. 자기 비평(self-critique) → 반복 개선 루프의 원리 이해
2. 품질 점수가 목표치에 도달할 때까지 답변을 스스로 개선하는 에이전트를 **실제 LLM 으로** 구현
3. **LLM 자기채점의 편향**과, 비평을 **검증기(verifier)에 근거**시켜 이를 보완하는 방법 이해
4. Self-Refine 접근이 거버넌스(품질 통제)에 기여하는 방식 이해

> 📦 **환경 설치·실행 명령**은 [`env_guides/M03_3_self_refine.md`](env_guides/M03_3_self_refine.md) 에 정리되어 있습니다.
> 반복·공통 구현은 [`agentic_lib/governance.py`](agentic_lib/governance.py)(`SelfCorrectingAgent`) 로 분리해 두었습니다.

> 참고 논문: *Self-Refine: Iterative Refinement with Self-Feedback* ([arXiv:2303.17651](https://arxiv.org/abs/2303.17651)).

---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가
import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import uv_install, get_llm, test_llm_connection, LLM_PROVIDER

# 반복/공통 거버넌스·트레이싱 구현은 agentic_lib 라이브러리로 분리되어 있습니다.
from agentic_lib import bootstrap, governance
from agentic_lib.bootstrap import to_text  # 공급자 무관 응답 정규화(<think>/list 제거)
from agentic_lib.governance import (
    SelfCorrectingAgent,
)

# 필요한 추가 의존성 설치 (uv → 실패 시 pip)
uv_install(['langchain', 'python-dotenv'])

llm = get_llm()  # 기본 공급자(ollama/qwen3:8b) LangChain BaseChatModel 반환
print("setup 완료 — LLM 공급자:", LLM_PROVIDER)

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct
[uv] 설치 완료: ['langchain', 'python-dotenv']
setup 완료 — LLM 공급자: nvidia


---
## 7. 자기 수정(Self-Correction) 메커니즘

**Self-Refine** 의 핵심은 하나의 반복 루프입니다.

> **생성(Generate) → 비평(Critique) → 개선(Refine) → …**  목표 품질에 도달하거나 최대 반복 횟수에 도달할 때까지 반복

- **Generate**: 과제에 대한 초안을 만든다.
- **Critique(자기 비평)**: 답변의 문제점을 스스로 진단하고 개선 방향을 제안한다. *이 비평의 질이 전체 성능을 좌우한다.*
- **Refine**: 비평을 반영해 답변을 다시 쓴다.

루프 구조는 같지만 **"점수를 누가 매기는가"** 에 따라 신뢰도가 크게 달라집니다. 아래에서 두 방식을 차례로 비교합니다.

| | 7.1 순수 자기채점 | 7.2 검증기 근거(grounded) |
|---|---|---|
| 점수 출처 | **LLM 이 루브릭으로 스스로 채점** | **코드 검증기**가 계산한 제약 만족 비율 |
| 장점 | 어떤 과제에도 적용 가능 | 점수가 객관적·재현 가능 |
| 약점 | 점수가 후해지는 편향·과신 | 검증 가능한 과제에만 적용 가능 |

### 7.1 순수 자기채점(self-scoring) Self-Refine — LLM 이 스스로 매긴 점수로 도는 루프

`agentic_lib.governance.SelfCorrectingAgent` 는 세 단계가 **모두 실제 LLM 호출**로 동작합니다.

| 메서드 | 하는 일 |
|---|---|
| `generate(task)` | 초안을 LLM 으로 생성 |
| `critique(task, response)` | **루브릭**(구체성·구조·실행가능성·정확성)을 받은 LLM 이 `{quality_score, issues, suggestions}` 를 **JSON 으로** 반환 |
| `refine(task, response, critique)` | 문제점·제안을 지침으로 주고 LLM 이 답변을 재작성 |
| `run(task, target_quality)` | 목표 점수 도달 또는 최대 반복까지 위 루프를 반복 |

두 가지 설계 포인트:
- **채점 축을 루브릭으로 고정**했습니다. 기준 없이 "점수를 매겨라" 하면 매번 다른 '느낌 점수'가 나옵니다.
- **생성용과 비평용 모델을 분리**합니다. 생성은 `temperature=0.5`(반복마다 답이 바뀔 다양성),
  비평은 `temperature=0`(채점 흔들림 최소화).

> ⚠️ JSON 파싱은 실패할 수 있습니다(공급자에 따라 ```json 펜스·`<think>`·설명이 섞여 나옴).
> `_parse_critique()` 가 첫 JSON 블록만 뽑고, 실패 시 중간 점수로 폴백해 루프가 죽지 않게 합니다.

In [2]:
# SelfCorrectingAgent(생성→자기 비평→개선) 구현은 agentic_lib.governance 에 있습니다.
# 참고: 'Self-Refine: Iterative Refinement with Self-Feedback' (arXiv:2303.17651)
sc_agent = SelfCorrectingAgent(
    llm=get_llm(temperature=0.5),        # 생성·개선용: 반복마다 답이 달라질 다양성
    critic_llm=get_llm(temperature=0),   # 비평용: 채점을 흔들리지 않게 고정
    max_iterations=3,                     # 비용 상한(반복 1회 = LLM 호출 2회)
)

# 과제는 '모델이 확실히 아는 주제'로 고른다. 모르는 주제를 주면 개선이 아니라 환각이 늘어난다.
# 분량 제한(5단계·2문장)은 반복할수록 답이 길어지기만 하는 현상을 억제한다.
SELF_SCORE_TASK = (
    "신입 개발자에게 Git 브랜치를 만들고 병합하는 절차를 설명하라. "
    "5단계 이내, 각 단계는 2문장 이내로 쓴다."
)
result = sc_agent.run(SELF_SCORE_TASK, target_quality=0.85)

print(f"\n{'='*60}\n[최종 답변] (개선 {result['iterations']-1}회, 자기채점 {result['final_score']:.2f})")
print(result["final_response"])


=== 자기 수정 에이전트 (목표 0.85, 최대 3회) ===
과제: 신입 개발자에게 Git 브랜치를 만들고 병합하는 절차를 설명하라. 5단계 이내, 각 단계는 2문장 이내로 쓴다.

[반복 1] 품질 점수: 0.60
  문제점: 1. 브랜치 생성 단계에서 명령어를 사용하는 순서가 논리적이지 않습니다. 브랜치를 생성하고 checkout하는 명령어를 분리해야 합니다., 3. 커밋 단계에서 커밋 메시지의 예시가 부족합니다. 커밋 메시지의 예시를 추가해야 합니다., 4. 원본 브랜치 병합 단계에서 merge 명령어를 사용하기 전에 checkout 명령어를 사용하는 것이 불필요합니다. checkout 명령어를 제거해야 합니다.
  개선 방향: 1. 브랜치 생성 단계에서 명령어를 사용하는 순서를 논리적으로 정리해야 합니다. 예를 들어, 브랜치를 생성한 후 checkout 명령어를 사용하는 것이 좋습니다., 2. 작업 수행 단계에서 git add 명령어를 사용할 때, 파일 이름 대신 '*'을 사용하여 모든 파일을 추가하는 방법을 설명해야 합니다., 3. 커밋 단계에서 커밋 메시지의 예시를 추가해야 합니다. 예를 들어, 'feat: 기능 추가'나 'fix: 버그 수정'과 같은 예시를 추가해야 합니다., 4. 원본 브랜치 병합 단계에서 merge 명령어를 사용하기 전에 checkout 명령어를 사용하는 것이 불필요합니다. checkout 명령어를 제거해야 합니다., 5. 병합 확인 단계에서 git log 명령어를 사용할 때, 특정 브랜치의 로그를 확인하는 방법을 설명해야 합니다. 예를 들어, 'git log master'와 같은 예시를 추가해야 합니다.
  → 개선본: 1. **브랜치 생성**: 새로운 브랜치를 생성하기 위해 `git branch <브랜치 이름>` 명령어를 사용합니다. 2. **브랜치 전환**:...

[반복 2] 품질 점수: 0.60
  문제점: 1단계에서 브랜치 이름을 지정하지 않았습니다., 3단계에서 '*'을 사용하여 모든 파일을 추가하는 것은 권장

**출력에서 확인할 것 — 그리고 이 방식의 한계**

`[점수 추이]` 를 먼저 보세요. 올라갔다면(예: `0.62 → 0.78 → 0.88`) 자기 비평이 답변을 실제로 밀어올린 것입니다.
하지만 **점수를 매긴 주체가 답을 쓴 주체와 같다**는 점 때문에 다음 중 하나가 자주 관찰됩니다.

| 실패 모드 | 증상 | 왜 문제인가 |
|---|---|---|
| **자기채점 인플레이션** | 초안부터 0.9 를 줘 **1회도 개선하지 않고** 종료 | 목표 도달이 아니라 자기 확신으로 멈춘 것 |
| **점수 발산·비단조** | `0.60 → 0.60 → 0.40` 처럼 오르내리며 수렴하지 않음 | 같은 답을 매번 다르게 채점 = 채점 기준이 불안정 |
| **장황화(verbosity drift)** | 반복할수록 내용은 그대로인데 길이만 늘어남 | 비평이 "더 자세히"만 요구할 때 생김 |
| **환각 증폭** | 모델이 잘 모르는 주제에서 "구체적으로 쓰라"는 비평에 **없는 사실을 지어내 채움** | 개선이 아니라 오염 |

> ⚠️ 특히 소형 모델(8B급)에서 두드러집니다. 위 코드에서 과제를 *모델이 확실히 아는 주제*로 고르고
> *분량 제한*을 건 이유가 뒤 두 실패 모드를 막기 위해서입니다.
> (모델이 모르는 주제로 바꿔 실행해 보면 환각 증폭을 직접 관찰할 수 있습니다.)

**결론**: 이 점수는 실제 품질이 아니라 **모델의 자기 확신**일 수 있습니다. 종료 조건을 여기에 맡기는 것은 위험합니다.
→ 그래서 다음 절에서는 점수의 근거를 **모델 바깥의 검증기**로 옮깁니다.

### 7.2 검증기에 근거한(grounded) 자기 수정 — 점수를 모델 바깥으로 옮기기

7.1 의 루프는 그대로 두고 **점수의 출처만 바꿉니다.** LLM 의 자기채점 대신
*코드로 검증 가능한 신호*를 쓰기 위해, *한 번에 만족시키기 어려운 다중 제약 생성(constrained generation)*
과제를 예로 듭니다. (Self-Refine 논문의 대표 과제 유형 중 하나입니다.)

**왜 이 과제가 자기 수정을 잘 보여주는가?**
- 제약이 여러 개라 **한 번(one-shot)에 모두 지키기 어렵다** → 초안은 거의 항상 일부를 위반한다.
- 위반 여부를 **코드로 객관적으로 검증**할 수 있다 → 비평(critique)을 *모델의 주관적 자기채점*이 아니라
  *검증기(verifier)의 객관적 신호*에 **근거(grounding)** 시킬 수 있다. 근거 있는 피드백이 자기 수정을 훨씬 안정적으로 만든다.

아래에서는 `generate` / `critique` / `refine` / 루프를 **노트북에서 직접 손으로 구현**해 각 부품이 무엇을 주고받는지 봅니다.
(같은 일을 라이브러리로 하려면 `SelfCorrectingAgent(llm=..., verifier=...)` 처럼 검증기를 주입하면 됩니다 — 정리 절 참고.)

> **핵심 교훈**: 자기 비평은 *검증 가능한 신호*(테스트 통과 여부, 제약 만족 여부 등)에 연결할수록 신뢰도가 올라갑니다.
> 순수하게 "LLM 이 스스로를 채점" 하는 방식(7.1)은 편향·과신에 취약합니다.

In [3]:
import re

# 다중 제약 생성 과제: 아래 5개 제약을 '모두' 만족하는 한국어 홍보 문구를 작성해야 한다.
TASK = (
    "'LLM 관측성(Observability)' 도구를 홍보하는 한국어 문구를 작성하라.\n"
    "다음 제약을 모두 지켜야 한다:\n"
    "  1) 전체 길이가 공백 포함 40자 이상 160자 이하\n"
    "  2) '관측' 과 '신뢰' 라는 단어를 모두 포함\n"
    "  3) 숫자를 '정확히 하나만' 포함 (예: 24 는 가능, 95% 와 100 처럼 두 개는 불가)\n"
    "  4) 정확히 2개의 문장 (각 문장은 마침표 '.' 로 끝남)\n"
    "  5) 물음표('?') 와 느낌표('!') 를 쓰지 않는다\n"
    "설명 없이 문구만 출력하라."
)

def count_sentences(text: str) -> int:
    """마침표+공백(또는 문장 끝)을 문장 경계로 세어 문장 수를 구한다.

    '99.9%' 처럼 소수점의 마침표는 뒤에 공백이 없으므로 문장 경계로 세지 않는다.
    """
    return len(re.findall(r'\.(?:\s|$)', text.strip()))

def check_constraints(text: str) -> dict:
    """과제의 5개 제약을 '코드로 객관적으로' 검사한다(자기 비평의 근거가 되는 검증기).

    각 위반은 '무엇이 왜 틀렸는지'를 수치로 설명하는 문자열로 돌려줘, refine 이 구체적으로 고치게 한다.
    """
    text = text.strip()
    n = len(text)
    ns = count_sentences(text)
    nums = re.findall(r'\d+', text)               # 텍스트에 등장한 '숫자 덩어리' 목록
    has_gwan, has_sin = ('관측' in text), ('신뢰' in text)

    results = [  # (통과여부, 위반 시 설명)
        (40 <= n <= 160,                          f"길이 위반: 현재 {n}자 (목표 40~160자)"),
        (has_gwan and has_sin,                    f"키워드 위반: '관측' 포함={has_gwan}, '신뢰' 포함={has_sin} (둘 다 필요)"),
        (len(nums) == 1,                          f"숫자 위반: 숫자를 정확히 하나만 포함해야 함 (현재 {len(nums)}개: {nums})"),
        (ns == 2 and text.endswith('.'),          f"문장 수 위반: 현재 {ns}문장 (정확히 2문장, 마침표로 끝나야 함)"),
        (('?' not in text) and ('!' not in text), "문장부호 위반: '?' 또는 '!' 를 쓰지 말 것"),
    ]
    checks = {"길이": results[0][0], "키워드": results[1][0], "숫자": results[2][0],
              "2문장": results[3][0], "부호": results[4][0]}
    violations = [msg for ok, msg in results if not ok]
    return {"checks": checks, "violations": violations, "length": n, "sentences": ns}

# 검증기 동작 확인 (통과 예 / 위반 예)
good = "관측 데이터로 신뢰를 지키는 24시간 모니터링. 오늘 팀의 관측 문화를 바꾸세요."
bad = "최고의 관측 도구!"
print(f"통과 예({len(good)}자) 위반:", check_constraints(good)["violations"])
print(f"위반 예({len(bad)}자) 위반:", check_constraints(bad)["violations"])

통과 예(45자) 위반: []
위반 예(10자) 위반: ['길이 위반: 현재 10자 (목표 40~160자)', "키워드 위반: '관측' 포함=True, '신뢰' 포함=False (둘 다 필요)", '숫자 위반: 숫자를 정확히 하나만 포함해야 함 (현재 0개: [])', '문장 수 위반: 현재 0문장 (정확히 2문장, 마침표로 끝나야 함)', "문장부호 위반: '?' 또는 '!' 를 쓰지 말 것"]


In [4]:
# 자기 수정에는 매 반복 답변이 바뀔 다양성이 필요하므로, 온도를 조금 높인 LLM 을 따로 준비한다.
gen_llm = get_llm(temperature=0.5)

def _clean(text: str) -> str:
    """LLM 이 덧붙인 따옴표/코드펜스/앞뒤 공백을 제거해 순수 문구만 남긴다."""
    t = text.strip().strip('`').strip()
    return t.strip('"').strip("'").strip()

def generate(task: str) -> str:
    """초안(초기 답변)을 LLM 으로 생성한다. (SelfCorrectingAgent.generate() 와 같은 역할)"""
    return _clean(bootstrap.invoke_text(gen_llm, task))

draft = generate(TASK)
print("[초안]")
print(draft)
print("\n[검증기 판정] 위반:", check_constraints(draft)["violations"])

[초안]
LLM 관측성(Observability) 도구는 시스템의 관측 가능성을 신뢰할 수 있는 방식으로 측정합니다. 
이 도구는 24 개의 주요 지표를 분석하여 시스템의 관측 가능성을 신뢰할 수 있는 방식으로 측정합니다. 
관측성은 시스템의 성능과 신뢰성을 높일 수 있는 중요한 지표입니다.

[검증기 판정] 위반: ['문장 수 위반: 현재 3문장 (정확히 2문장, 마침표로 끝나야 함)']


#### 핵심요소 ① Critique — 자기 비평 (검증기 근거 + LLM 제안)

`critique` 는 7.1 과 **동일한 형태의 딕셔너리** `{quality_score, issues, suggestions}` 를 돌려주되, 채우는 방식이 다릅니다.

| 필드 | 7.1 (자기채점) | 7.2 (검증기 근거) |
|---|---|---|
| `quality_score` | LLM 이 루브릭으로 매긴 점수 | **검증기**가 계산한 제약 만족 비율(0~1) |
| `issues` | LLM 이 지목한 문제점 | **검증기**가 찾은 구체적 위반 목록 |
| `suggestions` | LLM 의 개선 제안 | **LLM** 의 개선 제안 (동일) |

즉 *객관 신호로 점수·문제점의 근거를 잡고, 개선 아이디어는 LLM 이 낸다*는 하이브리드 비평입니다.
"무엇이 틀렸는지"는 기계가 판정하고, "어떻게 고칠지"는 모델이 제안하는 역할 분담입니다.

In [5]:
def critique(task: str, response: str) -> dict:
    """자기 비평: 검증기(객관)로 점수·문제점을 잡고, 개선 제안은 LLM 이 낸다.

    7.1 의 SelfCorrectingAgent.critique() 와 같은 {quality_score, issues, suggestions}
    형태를 돌려주지만, 점수는 LLM 자기채점이 아니라 검증기가 계산한다.
    """
    verdict = check_constraints(response)
    total = len(verdict["checks"])
    passed = total - len(verdict["violations"])
    quality_score = passed / total              # 객관적 점수: 만족한 제약의 비율

    issues = verdict["violations"]              # 객관적 문제점(위반 제약 목록)
    suggestions = []
    if issues:
        # 위반 목록을 주고 '어떻게 고칠지'만 LLM 에게 물어 자기 피드백(제안)을 받는다
        ask = (
            f"[과제]\n{task}\n\n[현재 답변]\n{response}\n\n"
            f"[검증기가 찾은 위반]\n- " + "\n- ".join(issues) + "\n\n"
            "각 위반을 고치기 위한 구체적 지침을 한 줄씩 bullet('- ')로만 제시하라. 다른 말은 하지 마라."
        )
        text = bootstrap.invoke_text(gen_llm, ask)
        suggestions = [ln.lstrip('-• ').strip() for ln in text.splitlines()
                       if ln.strip().startswith(('-', '•'))]

    return {"quality_score": quality_score, "issues": issues, "suggestions": suggestions}

crit = critique(TASK, draft)
print(f"품질 점수: {crit['quality_score']:.2f}  (만족 제약 비율 — 검증기 계산)")
print("문제점(issues):", crit["issues"])
print("개선 제안(suggestions):")
for s in crit["suggestions"]:
    print("  -", s)

품질 점수: 0.80  (만족 제약 비율 — 검증기 계산)
문제점(issues): ['문장 수 위반: 현재 3문장 (정확히 2문장, 마침표로 끝나야 함)']
개선 제안(suggestions):
  - 문장 수를 2개로 제한하기 위해 두 번째 문장을 첫 번째 문장과 합치시오.
  - 문장의 내용을 재구성하여 '관측' 과 '신뢰' 라는 단어를 모두 포함하되, '관측성은' 부분을 삭제하시오.


#### 핵심요소 ② Refine — 비평을 반영한 재작성

`refine` 은 *원래 과제 + 직전 답변 + 비평(문제점·제안)* 을 LLM 에게 함께 주고 **더 나은 답변을 다시 쓰게** 합니다.
비평이 *구체적일수록* 개선이 잘 됩니다. 아래에서 한 번의 개선으로 위반이 줄어드는지 확인합니다.

In [6]:
def refine(task: str, response: str, crit: dict) -> str:
    """비평(문제점+제안)을 반영해 답변을 LLM 으로 다시 쓴다. (7.1 의 refine() 과 같은 역할)"""
    if not crit["issues"]:
        return response                          # 고칠 게 없으면 그대로 반환
    # 검증기의 '수치화된 위반'(issues) + LLM 의 개선 제안(suggestions)을 모두 지침으로 준다
    guide = "\n".join([f"- {i}" for i in crit["issues"]] +
                      [f"- {s}" for s in crit["suggestions"]])
    prompt = (
        f"[과제]\n{task}\n\n"
        f"[직전 답변]\n{response}\n\n"
        f"[반드시 고칠 점]\n{guide}\n\n"
        "위 지침을 모두 반영해, 과제의 제약을 전부 지키는 새 문구만 출력하라. 설명은 하지 마라."
    )
    return _clean(bootstrap.invoke_text(gen_llm, prompt))

improved = refine(TASK, draft, crit)
print("[1차 개선본]")
print(improved)
print("\n[검증기 판정] 위반:", check_constraints(improved)["violations"])

[1차 개선본]
LLM 관측성(Observability) 도구는 시스템의 관측 가능성을 신뢰할 수 있는 방식으로 측정합니다. 이 도구는 시스템의 성능과 신뢰성을 높일 수 있는 중요한 지표인 24 개의 주요 지표를 분석하여 신뢰할 수 있는 관측 가능성을 측정합니다.

[검증기 판정] 위반: []


#### 핵심요소 ③ Loop — 목표 품질에 도달할 때까지 반복

세 요소를 하나로 묶습니다. 7.1 의 `SelfCorrectingAgent.run()` 과 **같은 구조**이며(라이브러리에 감춰져 있던
루프를 여기서는 펼쳐서 봅니다), 점수만 검증기에서 옵니다.
매 반복마다 점수·위반을 출력하고, **모든 제약을 만족(점수 1.0)** 하거나 **최대 반복 횟수**에 도달하면 멈춥니다.

7.1 과 달리 목표를 `1.0`(모든 제약 만족)으로 둘 수 있다는 점에 주목하세요.
점수가 객관적이므로 **"만족했다"는 판정을 그대로 믿고 종료 조건으로 쓸 수 있습니다.**

In [7]:
def run_self_refine(task: str, target_quality: float = 1.0, max_iterations: int = 5) -> dict:
    """생성 → (비평 → 개선) 반복. SelfCorrectingAgent.run() 의 검증기 근거 버전.

    Args:
        task: 처리할 과제.
        target_quality: 도달 목표 점수(1.0 = 모든 제약 만족).
        max_iterations: 최대 개선 반복 횟수.

    Returns:
        최종 답변/반복 횟수/남은 위반/이력을 담은 dict.
    """
    print(f"=== 자기 수정 시작 (목표 점수 {target_quality}, 최대 {max_iterations}회) ===")
    response = generate(task)
    crit = critique(task, response)                        # 초안에 대한 첫 비평
    history = [{"iteration": 0, "response": response, "score": crit["quality_score"]}]

    for i in range(1, max_iterations + 1):
        print(f"\n[반복 {i}] 품질 점수: {crit['quality_score']:.2f}  ·  위반: {crit['issues'] or '없음'}")

        if crit["quality_score"] >= target_quality:
            print("  ✅ 목표 품질 달성 — 모든 제약 만족, 반복 종료")
            break

        response = refine(task, response, crit)            # 비평 반영 재작성
        crit = critique(task, response)                    # 개선본을 다시 비평
        history.append({"iteration": i, "response": response, "score": crit["quality_score"]})
        print(f"  → 개선본: {response[:80]}")
    else:
        print("\n  ⏹ 최대 반복 도달 — 자기 수정은 개선을 돕지만 완벽을 보장하진 않는다(과도한 반복은 비용↑)")

    final_violations = check_constraints(response)["violations"]
    print(f"\n[점수 추이] " + " → ".join(f"{h['score']:.2f}" for h in history))
    print(f"\n[최종 답변] (남은 위반 {len(final_violations)}개, 개선 {len(history)-1}회)")
    print(response)
    return {"final_response": response, "iterations": len(history),
            "final_violations": final_violations, "history": history}

result = run_self_refine(TASK)

=== 자기 수정 시작 (목표 점수 1.0, 최대 5회) ===

[반복 1] 품질 점수: 0.60  ·  위반: ['길이 위반: 현재 219자 (목표 40~160자)', '문장 수 위반: 현재 3문장 (정확히 2문장, 마침표로 끝나야 함)']
  → 개선본: LLM 관측성(Observability) 도구는 시스템의 상태를 신뢰할 수 있는 방식으로 관측하고 관리하는 데 도움을 줄 수 있습니다. 시스템의

[반복 2] 품질 점수: 0.60  ·  위반: ['길이 위반: 현재 200자 (목표 40~160자)', '문장 수 위반: 현재 3문장 (정확히 2문장, 마침표로 끝나야 함)']
  → 개선본: LLM 관측성(Observability) 도구는 시스템의 상태를 신뢰할 수 있는 방식으로 관측하고 관리하는 데 도움을 줄 수 있습니다. 시스템의

[반복 3] 품질 점수: 0.80  ·  위반: ['숫자 위반: 숫자를 정확히 하나만 포함해야 함 (현재 0개: [])']
  → 개선본: LLM 관측성(Observability) 도구는 시스템의 상태를 신뢰할 수 있는 방식으로 24시간 내에 관측하고 관리하는 데 도움을 줄 수 있습

[반복 4] 품질 점수: 1.00  ·  위반: 없음
  ✅ 목표 품질 달성 — 모든 제약 만족, 반복 종료

[점수 추이] 0.60 → 0.60 → 0.80 → 1.00

[최종 답변] (남은 위반 0개, 개선 3회)
LLM 관측성(Observability) 도구는 시스템의 상태를 신뢰할 수 있는 방식으로 24시간 내에 관측하고 관리하는 데 도움을 줄 수 있습니다. 시스템의 상태를 신뢰할 수 있는 방식으로 관측하고, 문제가 발생할 경우 신속하게 대응할 수 있는 시스템을 구축할 수 있습니다.


---
## 정리

1. **자기 수정 루프:** 생성(Generate) → 비평(Critique) → 개선(Refine) 을 목표 품질/최대 반복까지 반복한다.
2. **핵심은 '비평'의 질:** 비평이 구체적일수록 개선이 잘 된다. 비평을 **검증 가능한 신호(테스트·제약 검증기)에 근거**시키면
   순수 LLM 자기채점보다 훨씬 안정적이다(grounding).
3. **자기채점의 함정(7.1 → 7.2):** 답을 쓴 모델이 그 답을 채점하면 점수가 후해지고 단조 증가한다.
   *점수를 신뢰할 수 없으면 종료 조건도 신뢰할 수 없다.* 검증 가능한 과제라면 점수를 모델 바깥으로 빼라.
4. **종료 조건:** 목표 품질 도달 또는 최대 반복 도달. 자기 수정은 *개선*을 돕지만 *완벽*을 보장하진 않는다(과도한 반복은 비용↑).
   반복 1회당 LLM 호출이 2회 늘어난다는 점도 설계에 반영해야 한다.
5. **거버넌스 관점:** 출력 품질을 시스템적으로 끌어올리는 통제 장치 — 가드레일(안전)·트레이싱(관측)과 상호 보완.

### 라이브러리로 재사용하기

7.2 에서 손으로 짠 루프는 `SelfCorrectingAgent` 에 **검증기를 주입**하면 그대로 대체됩니다.

```python
def constraint_verifier(text: str):
    """검증기 어댑터: (점수, 위반목록) 을 돌려주면 자기채점 대신 이 값이 쓰인다."""
    verdict = check_constraints(text)
    total = len(verdict["checks"])
    return (total - len(verdict["violations"])) / total, verdict["violations"]

agent = SelfCorrectingAgent(llm=gen_llm, max_iterations=5, verifier=constraint_verifier)
result = agent.run(TASK, target_quality=1.0)
```

`verifier` 를 빼면 7.1 의 순수 자기채점, 넣으면 7.2 의 근거 있는 비평 — **루프는 하나, 점수의 출처만 바뀝니다.**

### 참고 자료
- Self-Refine 논문: https://arxiv.org/abs/2303.17651